# Demo — End-to-End Single-File Walkthrough

Preprocess one audio file, run PR + G2P, and compute edit distance.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import soundfile as sf

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path("../../../mod").resolve()))

from config import DATA_DIR, DEVICE, LANG, MODEL_NAME, USE_FLASH_ATTN
from powsm.espnet_subsampling_prefix_compat import apply_subsampling_prefix_compat
from powsm.ipa import cmu_target_ipa, tokenize_ipa
from assessment.edit_distance import edit_operations

apply_subsampling_prefix_compat()
from espnet2.bin.s2t_inference_ctc import Speech2TextGreedySearch

model = Speech2TextGreedySearch.from_pretrained(
    MODEL_NAME, device=DEVICE, use_flash_attn=USE_FLASH_ATTN,
    lang_sym=LANG, task_sym="<pr>",
)
print("Model loaded.")

In [ ]:
# Pick a file
AUDIO = DATA_DIR / "12" / "umit12.wav"
TEXT = (DATA_DIR / "12" / "text").read_text().strip()

speech, rate = sf.read(str(AUDIO))
print(f"Audio: {AUDIO.name}  ({len(speech)/rate:.1f}s @ {rate} Hz)")
print(f"Text:  {TEXT}")

In [ ]:
# Phone Recognition
audio = np.squeeze(speech).astype(np.float32)
raw = model.batch_decode([audio], batch_size=1)[0]
if "<notimestamps>" in raw:
    raw = raw.split("<notimestamps>")[1]
pr_phones = [p.strip().strip("/") for p in raw.strip().split("//") if p.strip().strip("/")]
print(f"PR ({len(pr_phones)} phones): {raw.strip()[:100]}...")

# CMU target
target_ipa = cmu_target_ipa(TEXT)
target_tokens = tokenize_ipa(target_ipa)
print(f"CMU target ({len(target_tokens)} phones): {target_ipa[:100]}...")

In [ ]:
# Assessment
ops = edit_operations(pr_phones, target_tokens)

ins = sum(1 for o in ops if o[0] == "insert")
dels = sum(1 for o in ops if o[0] == "delete")
subs = sum(1 for o in ops if o[0] == "substitute")

print(f"Total errors: {len(ops)}")
print(f"  Insertions:    {ins}")
print(f"  Deletions:     {dels}")
print(f"  Substitutions: {subs}")
print(f"\nFirst 10 operations:")
for op in ops[:10]:
    print(f"  {op}")